# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend sons based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [1]:
!pip install librosa
!pip install panns-inference
!pip install torchcodec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.0 MB/s eta 0:00:00


In [2]:
!pip install -qU pinecone-client==3.1.0 panns-inference datasets librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.0/211.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 54.9 MB/s eta 0:00:00


# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [3]:
from datasets import load_dataset

# load the dataset from huggingface model hub
data = load_dataset("sanchit-gandhi/gtzan", split="train")
data

README.md:   0%|          | 0.00/703 [00:00<?, ?B/s]

data/train-00000-of-00003-abaaa5719027ce(…):   0%|          | 0.00/441M [00:00<?, ?B/s]

data/train-00001-of-00003-40e2de07ad4288(…):   0%|          | 0.00/429M [00:00<?, ?B/s]

data/train-00002-of-00003-6e2eb838540a06(…):   0%|          | 0.00/436M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/999 [00:00<?, ? examples/s]

Dataset({
    features: ['file', 'audio', 'genre'],
    num_rows: 999
})

In [4]:
data

Dataset({
    features: ['file', 'audio', 'genre'],
    num_rows: 999
})

The audios in the dataset are sampled at 44100Hz and loaded into NumPy arrays. Let's take a look.

In [5]:
# select the audio feature and display top three
audios = data["audio"]
audios[:3]

We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [6]:
import numpy as np

# select only the audio data from the dataset and store in a list of numpy arrays
audios = [a["array"] for a in data["audio"]]

# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [7]:
from panns_inference import AudioTagging

# load the default model into the gpu.
model = AudioTagging(checkpoint_path=None, device='mps') # change device to cpu if a gpu is not available

Checkpoint path: /root/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


## Initializing the Index

Now we need a place to store these embeddings and enable a efficient vector search through them all. To do that we use Pinecone, we can get a [free API key](https://app.pinecone.io/) and enter it below where we will initialize our connection to Pinecone and create a new index.

In [8]:
import os
from getpass import getpass

# Prompt for OpenAI API key without echoing input
os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key: ")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Prompt for Pinecone API key without echoing input
os.environ["PINECONE_API_KEY"] = getpass("Enter Pinecone API key: ")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")


Enter OpenAI API key: ··········
Enter Pinecone API key: ··········


In [9]:
import os
from pinecone import Pinecone

# configure client
pc = Pinecone(api_key=PINECONE_API_KEY)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [10]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Create the index:

In [11]:
index_name = "audio-search-demo"

In [12]:
import time

# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=2048,
        metric='cosine',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
# view index stats
index.describe_index_stats()

{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 2000}},
 'total_vector_count': 2000}

First, let's see what indexes you currently have in your Pinecone account.

In [13]:
import os
from pinecone import Pinecone

# Ensure PINECONE_API_KEY is set
if not os.getenv("PINECONE_API_KEY"):
    print("PINECONE_API_KEY not found. Please run the cell to set API keys first.")
else:
    pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
    print("Your existing Pinecone indexes:")
    print(pc.list_indexes().names())


Your existing Pinecone indexes:
['abstractive-question-answering', 'langchain-retrieval-agent', 'audio-search-demo']


From the list above, choose an index you no longer need and run the following cell to delete it. Replace `"YOUR_INDEX_TO_DELETE"` with the actual name of the index you wish to delete. After deleting an index, you can then re-run the original index creation cell to successfully create `audio-search-demo`.

In [14]:
# Replace "YOUR_INDEX_TO_DELETE" with the name of an actual index from the list above.
index_to_delete = "question-answering"

if index_to_delete and index_to_delete in pc.list_indexes().names():
    pc.delete_index(index_to_delete)
    print(f"Index '{index_to_delete}' deleted successfully.")
else:
    print(f"Index '{index_to_delete}' not found or not specified for deletion.")

# Verify deletion
print("Your remaining Pinecone indexes:")
print(pc.list_indexes().names())

Index 'question-answering' not found or not specified for deletion.
Your remaining Pinecone indexes:
['abstractive-question-answering', 'langchain-retrieval-agent', 'audio-search-demo']


# Generate Embeddings and Upsert

Now we generate the embeddings using the audio embedding model. We must do this in batches as processing all items at once will exhaust machine memory limits and API request limits.

In [16]:
from tqdm.auto import tqdm
import numpy as np # Ensure numpy is imported

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(audios))
    # extract batch
    batch = audios[i:i_end]

    # Find the maximum length in the current batch and pad shorter audios
    max_len = max(len(a) for a in batch)
    padded_batch = []
    for audio_arr in batch:
        if len(audio_arr) < max_len:
            pad_width = max_len - len(audio_arr)
            padded_audio = np.pad(audio_arr, (0, pad_width), 'constant')
            padded_batch.append(padded_audio)
        else:
            padded_batch.append(audio_arr)

    # generate embeddings for all the audios in the batch
    _, emb = model.inference(np.stack(padded_batch))
    # create unique IDs
    ids = [f"{idx}" for idx in range(i, i_end)]
    # add all to upsert list
    to_upsert = list(zip(ids, emb.tolist()))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/16 [00:00<?, ?it/s]

{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 2000}},
 'total_vector_count': 2000}

We now have *2000* audio records indexed in Pinecone, we're ready to begin querying.

# Querying

Let's first listen to an audio from our dataset. We will generate embeddings for the audio and use it to find similar audios from the Pinecone index.

In [33]:
from IPython.display import Audio, display

# we set an audio number to select from the dataset
audio_num = 300
# get the audio data of the audio number
query_audio = data[audio_num]["audio"]["array"]
# get the category of the audio number
category = data[audio_num]["genre"]
# print the category and play the audio
print("Query Audio:", category)
Audio(query_audio, rate=44100)

Query Audio: 3


We have got the sound of a car horn. Let's generate an embedding for this sound.

In [34]:
# reshape query audio
query_audio = query_audio[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)
xq.shape

(1, 2048)

We have now converted the audio into a 2048-dimension vector the same way we did for all the other audio we indexed. Let's use this to query our Pinecone index.

In [35]:
# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)
results

{'matches': [{'id': '300', 'score': 0.999255419, 'values': []},
             {'id': '448', 'score': 0.837596, 'values': []},
             {'id': '365', 'score': 0.821846247, 'values': []}],
 'namespace': '',
 'usage': {'read_units': 1}}

Notice that the top result is the audio number 400 from our dataset, which is our query audio (the most similar item should always be the query itself). Let's listen to the top three results.

In [36]:
# play the top 3 similar audios
for r in results["matches"]:
    # select the audio data from the databse using the id as an index
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

We have great results, everything aligns with what seems to be a busy city street with car horns.

Let's write a helper function to run the queries using audio from our dataset easily. We do not need to embed these audio samples again as we have already, they are just stored in Pinecone. So, we specify the `id` of the query audio to search with and tell Pinecone to search with that.

In [37]:
def find_similar_audios(id):
    print("Query Audio:")
    # select the audio data from the databse using the id as an index
    query_audio = data[id]["audio"]["array"]
    # play the query audio
    display(Audio(query_audio, rate=44100))
    # query pinecone index with the query audio id
    result = index.query(id=str(id), top_k=5)
    print("Result:")
    # play the top 5 similar audios
    for r in result["matches"]:
        a = data[int(r["id"])]["audio"]["array"]
        display(Audio(a, rate=44100))

In [39]:
find_similar_audios(300)

Query Audio:


Result:


Here we return a set of revving motors (they seem to either be vehicles or lawnmowers).

In [44]:
find_similar_audios(752)

Query Audio:


Result:


IndexError: Invalid key: 1732 is out of bounds for size 999

And now a more relaxing set of birds chirping in nature.

Let's use another audio sample from elsewhere (eg not this dataset) and see how the search performs with this.

In [51]:
!wget -O cat_sound.wav https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3

--2025-11-24 15:56:07--  https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3
Resolving www.soundhelix.com (www.soundhelix.com)... 81.169.145.157, 2a01:238:20a:202:1157::
Connecting to www.soundhelix.com (www.soundhelix.com)|81.169.145.157|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8945229 (8.5M) [audio/mpeg]
Saving to: ‘cat_sound.wav’

cat_sound.wav       100%[===================>]   8.53M  5.02MB/s    in 1.7s    

2025-11-24 15:56:10 (5.02 MB/s) - ‘cat_sound.wav’ saved [8945229/8945229]



We can load the audio into a Numpy array as follows:

In [52]:
import librosa
from IPython.display import Audio

a, _ = librosa.load("/content/cat_sound.wav", sr=44100)
Audio(a, rate=44100)

Now we generate the embeddings for this audio and query the Pinecone index.

In [53]:
# reshape query audio
query_audio = a[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)

# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)

# play the top 3 similar audios
for r in results["matches"]:
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

Our audio search application has identified a set of similar cat sounds, which is excellent.

# Delete the Index

Delete the index once you are sure that you do not want to use it anymore. Once the index is deleted, you cannot use it again.

In [ ]:
pc.delete_index(index_name)